<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/Step02bDetectFull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 02b · Detect cows with YOLO: full hour

**YOLO26n** looks at the whole clip of `ReferenceClip.mp4`, 2 frames per second, and draws a **box** around every cow it sees, with a confidence from 0 to 1. It does **not** link boxes over time; that is step 03 (SAM 2.1). Takes ~10–15 min on a T4.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step02bDetectFull/`, **overwriting** each run:

- `Detections.csv`: one row per box (Frame, TimeSeconds, X1..Y2, Confidence): **input for step 03b**
- `DetectionSummary.csv`: per minute of the clip, the share of frames with a cow (`CowShare`) and how sure YOLO was: shows when she is hard to see
- `Detect.json`: the settings used
- `PreviewDetections.mp4` (if `Preview = True`): boxes drawn, 4x speed

Needs a **T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Detect import DetectCows
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/
ClipPath = VideoOutputDir / "ReferenceClip.mp4"   # step 01
OutputDir = VideoOutputDir / "Step02bDetectFull"

StartSeconds = 0   # seconds into the clip (add 03:35:11 for video time)
EndSeconds = None   # None = to the end of the clip
TargetFps = 2          # the camera only updates ~2 times a second; step 03 follows the same frames
Confidence = 0.25      # lower = more boxes (and more false ones)
ImageSize = 1024       # YOLO's picture size; 640 (YOLO's default) misses the lying cow far more often
Preview = False   # True = also a preview video of the hour (slower)

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"

In [ ]:
# 3. Run: detect, then show the per-minute summary and the preview.
import pandas as pd
from IPython.display import Video, display

Stats = DetectCows(ClipPath, OutputDir, StartSeconds, EndSeconds, TargetFps=TargetFps, Confidence=Confidence,
                   ImageSize=ImageSize, Preview=Preview)

display(pd.read_csv(OutputDir / "DetectionSummary.csv"))
Path = OutputDir / "PreviewDetections.mp4"
if Path.exists():
    Mb = Path.stat().st_size / 2**20
    print(f"Preview: {Path} ({Mb:.0f} MB)")
    if Mb < 50:   # bigger files are too heavy to play inside the notebook; open them from Drive
        display(Video(str(Path), embed=True, width=960))

## About this step

**Model:** **YOLO26n** (Ultralytics, trained on the COCO dataset; only class 19 "cow" is kept). Not trained on our cows.

**How to read the output:** a box is "a cow is here in this frame"; it says nothing about *which* cow, and boxes are not linked over time. YOLO misses her in some frames (lying, straining, odd angles); step 03 fills those gaps.

**What we found:** image size matters. On 1080–1320 s, where she lies and strains, YOLO at size **640** found her in only **15 %** of frames (confidence ~0.3–0.5), at **1024** in **86 %**. Over the whole hour at 640: cow in 47 % of frames, 513 s on a T4 (14.5 frames/s). So the pipeline uses 1024 (a bit slower). Earlier, with ByteTrack, YOLO lost her for long stretches between ~630 and 2500 s while she lay and strained.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.